# Error Analysis for Turkish Legal RAG Experiments

This notebook analyzes the errors of the best pre-fine-tuning RAG pipeline:

Turkish BGE Reranker Fusion RAG

The goal is to identify whether failures are caused by:
- wrong retrieval context
- incomplete retrieval context
- correct context but wrong generation
- dataset mismatch
- partially correct generation

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
project_path = "/content/drive/MyDrive/turkish_legal_rag"
metrics_path = f"{project_path}/outputs/metrics"

print("Project path:", project_path)
print("Metrics path:", metrics_path)

Project path: /content/drive/MyDrive/turkish_legal_rag
Metrics path: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics


In [4]:
import os

print("Project exists:", os.path.exists(project_path))
print("Metrics exists:", os.path.exists(metrics_path))

Project exists: True
Metrics exists: True


In [5]:
!ls -lh "/content/drive/MyDrive/turkish_legal_rag/outputs/metrics"

total 410K
-rw------- 1 root root 1.3K Apr 27 18:11 baseline_retrieval_details.csv
-rw------- 1 root root   88 Apr 27 18:11 baseline_retrieval_keyword_metrics.csv
-rw------- 1 root root 2.7K Apr 27 22:50 base_rag_generation_results.csv
-rw------- 1 root root   71 Apr 27 22:50 base_rag_manual_score.csv
-rw------- 1 root root  17K Apr 27 23:06 base_rag_testset_generation_results.csv
-rw------- 1 root root 7.2K May  3 16:23 flashrank_fusion_reranker_4question_results.csv
-rw------- 1 root root  171 May  3 16:23 flashrank_fusion_reranker_4question_score.csv
-rw------- 1 root root  69K May  3 16:31 flashrank_fusion_reranker_rag_testset_generation_results.csv
-rw------- 1 root root  274 May  3 16:35 flashrank_fusion_reranker_rag_testset_score.csv
-rw------- 1 root root  69K May  3 16:35 flashrank_fusion_reranker_rag_testset_scored.csv
-rw------- 1 root root 1.6K Apr 27 18:46 hybrid_retrieval_details.csv
-rw------- 1 root root  155 Apr 27 18:46 hybrid_retrieval_keyword_metrics.csv
-rw------- 

In [6]:
import pandas as pd

df = pd.read_csv(
    f"{metrics_path}/turkish_bge_reranker_fusion_rag_testset_scored.csv"
)

df.head()

,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_context,top1_original_rank,top1_rerank_rank,top1_rerank_score,top1_fusion_score,retrieved_contexts,is_valid_sample,manual_score
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,Sen Türk hukuk metinleri için çalışan dikkatli...,"Anayasanın 101. Maddesiyle ilgili tartışmalar,...",chunk_000210,Türkiye Cumhuriyeti Anayasası,Madde 158 – Uyuşmazlık Mahkemesi adli ve idari...,1,4,0.007053,0.7750,Madde 158 – Uyuşmazlık Mahkemesi adli ve idari...,True,0.0
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...",Sen Türk hukuk metinleri için çalışan dikkatli...,"Anayasanın 10. Maddesi, herkesin eşittirliği s...",chunk_000188,Türkiye Cumhuriyeti Anayasası,"Madde 150 – Kanunların, Cumhurbaşkanlığı karar...",1,4,0.000305,0.7750,"Madde 150 – Kanunların, Cumhurbaşkanlığı karar...",True,0.5
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...",Sen Türk hukuk metinleri için çalışan dikkatli...,"Hayatın sınırlanması, Anayasanın 17. Maddesine...",chunk_000373,Türkiye Cumhuriyeti Anayasası,"Madde 17 – Herkes, yaşama, maddi ve manevi var...",1,1,0.385170,1.0000,"Madde 17 – Herkes, yaşama, maddi ve manevi var...",True,0.5
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,Sen Türk hukuk metinleri için çalışan dikkatli...,Geçici madde 20 1987 yılında eklendi.,chunk_000600,Bilgi Edinme Kanunu,Madde 20- Açıklanması veya zamanından önce açı...,1,8,0.000864,0.7375,Madde 20- Açıklanması veya zamanından önce açı...,True,0.0
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,Sen Türk hukuk metinleri için çalışan dikkatli...,"Hayır, TCK 121 ihlali sabit değildir. (Madde 1...",chunk_000101,Türkiye Cumhuriyeti Anayasası,Madde 121 – (Mülga: 21/1/2017-6771/16 md.) B. ...,1,6,0.000157,0.7500,Madde 121 – (Mülga: 21/1/2017-6771/16 md.) B. ...,True,0.5


In [7]:
print(df.shape)
print(df.columns.tolist())
df.head()

(20, 14)
['question', 'expected_answer', 'generated_answer', 'clean_generated_answer', 'top1_chunk_id', 'top1_source', 'top1_context', 'top1_original_rank', 'top1_rerank_rank', 'top1_rerank_score', 'top1_fusion_score', 'retrieved_contexts', 'is_valid_sample', 'manual_score']


,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_context,top1_original_rank,top1_rerank_rank,top1_rerank_score,top1_fusion_score,retrieved_contexts,is_valid_sample,manual_score
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,Sen Türk hukuk metinleri için çalışan dikkatli...,"Anayasanın 101. Maddesiyle ilgili tartışmalar,...",chunk_000210,Türkiye Cumhuriyeti Anayasası,Madde 158 – Uyuşmazlık Mahkemesi adli ve idari...,1,4,0.007053,0.7750,Madde 158 – Uyuşmazlık Mahkemesi adli ve idari...,True,0.0
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...",Sen Türk hukuk metinleri için çalışan dikkatli...,"Anayasanın 10. Maddesi, herkesin eşittirliği s...",chunk_000188,Türkiye Cumhuriyeti Anayasası,"Madde 150 – Kanunların, Cumhurbaşkanlığı karar...",1,4,0.000305,0.7750,"Madde 150 – Kanunların, Cumhurbaşkanlığı karar...",True,0.5
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...",Sen Türk hukuk metinleri için çalışan dikkatli...,"Hayatın sınırlanması, Anayasanın 17. Maddesine...",chunk_000373,Türkiye Cumhuriyeti Anayasası,"Madde 17 – Herkes, yaşama, maddi ve manevi var...",1,1,0.385170,1.0000,"Madde 17 – Herkes, yaşama, maddi ve manevi var...",True,0.5
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,Sen Türk hukuk metinleri için çalışan dikkatli...,Geçici madde 20 1987 yılında eklendi.,chunk_000600,Bilgi Edinme Kanunu,Madde 20- Açıklanması veya zamanından önce açı...,1,8,0.000864,0.7375,Madde 20- Açıklanması veya zamanından önce açı...,True,0.0
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,Sen Türk hukuk metinleri için çalışan dikkatli...,"Hayır, TCK 121 ihlali sabit değildir. (Madde 1...",chunk_000101,Türkiye Cumhuriyeti Anayasası,Madde 121 – (Mülga: 21/1/2017-6771/16 md.) B. ...,1,6,0.000157,0.7500,Madde 121 – (Mülga: 21/1/2017-6771/16 md.) B. ...,True,0.5


In [9]:
df["manual_score"].value_counts(dropna=False)

,count
manual_score,
0.0,10
0.5,8
1.0,2


In [10]:
valid_df = df[df["is_valid_sample"] == True].copy()

print("Valid sample count:", len(valid_df))
print("Mean score:", valid_df["manual_score"].mean())
print(valid_df["manual_score"].value_counts())

Valid sample count: 19
Mean score: 0.3157894736842105
manual_score
0.0    9
0.5    8
1.0    2
Name: count, dtype: int64


In [11]:
df["retrieval_quality"] = ""
df["generation_quality"] = ""
df["error_type"] = ""
df["notes"] = ""

In [12]:
retrieval_quality_labels = [
    "partial_context",     # 0 - 101. madde sorusu, cevap beklenen tartışmayı vermiyor.
    "correct_context",     # 1 - Eşitlik ilkesini yakalıyor ama cevap net değil.
    "correct_context",     # 2 - 17. madde/yaşama hakkı bağlamı var ama hukuki sonuç karışık.
    "wrong_context",       # 3 - Geçici madde 20 tarihi yanlış context.
    "wrong_context",       # 4 - TCK 121 yerine Anayasa 121/sıkıyönetim gibi yanlış bağlam.
    "wrong_context",       # 5 - Cumhurbaşkanı yemini yerine alakasız yemin/yazman cevabı.
    "wrong_context",       # 6 - Anayasa 122 içeriği yanlış.
    "correct_context",     # 7 - Yedi gün doğru.
    "correct_context",     # 8 - Madde 50 bağlamı var ama sonuç yanlış.
    "correct_context",     # 9 - Madde 108 bağlamı var ama Silahlı Kuvvetler istisnasını yanlış yorumluyor.
    "correct_context",     # 10 - Madde 13 bağlamı var, sonuç kısmen doğru.
    "wrong_context",       # 11 - KVKK ilgili kişi yerine CMK şüpheli/sanık bağlamı.
    "wrong_context",       # 12 - AYM karar kesinliği yerine üyeliğin sona ermesi.
    "correct_context",     # 13 - Madde 63 doğru.
    "partial_context",     # 14 - Madde 47 kısmen doğru bağlam ama beklenen cevap tam değil.
    "partial_context",     # 15 - Kişisel veriler için kanunla düzenleme var ama KVKK net değil.
    "partial_context",     # 16 - Madde 140 kısmen yakalanmış.
    "wrong_context",       # 17 - Ekonomik ve Sosyal Konsey / 115. madde yok.
    "partial_context",     # 18 - Kurul fikri var ama başvuru/ihlal giderimi eksik.
    "dataset_mismatch"     # 19 - invalid sample.
]

generation_quality_labels = [
    "wrong_generation",    # 0
    "partial_generation",  # 1
    "partial_generation",  # 2
    "wrong_generation",    # 3
    "partial_generation",  # 4
    "wrong_generation",    # 5
    "wrong_generation",    # 6
    "correct_generation",  # 7
    "wrong_generation",    # 8
    "wrong_generation",    # 9
    "partial_generation",  # 10
    "wrong_generation",    # 11
    "wrong_generation",    # 12
    "correct_generation",  # 13
    "partial_generation",  # 14
    "partial_generation",  # 15
    "partial_generation",  # 16
    "wrong_generation",    # 17
    "partial_generation",  # 18
    "not_applicable"       # 19
]

error_type_labels = [
    "partial_context",                    # 0
    "incomplete_answer",                  # 1
    "correct_context_wrong_generation",   # 2
    "wrong_context",                      # 3
    "wrong_context_partial_answer",       # 4
    "wrong_context",                      # 5
    "wrong_context",                      # 6
    "correct",                            # 7
    "correct_context_wrong_generation",   # 8
    "correct_context_wrong_generation",   # 9
    "partial_answer",                     # 10
    "wrong_context",                      # 11
    "wrong_context",                      # 12
    "correct",                            # 13
    "partial_answer",                     # 14
    "partial_answer",                     # 15
    "partial_answer",                     # 16
    "wrong_context",                      # 17
    "partial_answer",                     # 18
    "dataset_mismatch"                    # 19
]

notes = [
    "Expected discussion about Article 101 eligibility/application, but generated answer focused on elections/second voting.",
    "Equality principle is partially mentioned, but answer is unclear and does not directly state unconstitutionality.",
    "Mentions life/right limitation, but legal conclusion is confused compared to expected answer.",
    "Expected date is 20 May 2016, generated says 1987.",
    "Generated says violation is not fixed, but retrieved context is wrong and lacks expert report reasoning.",
    "Wrong context; answer confuses presidential oath with unrelated oath/writer issue.",
    "Wrong content for Article 122.",
    "Correctly answers seven days.",
    "Expected unconstitutional, generated says no.",
    "Expected DDK cannot inspect Armed Forces, generated says inspection is lawful.",
    "Says arbitrary limitation may be unconstitutional, but answer is weak/indirect.",
    "Wrong source/context; KVKK definition expected, CMK meaning generated.",
    "Wrong context; discusses Constitutional Court membership termination, not finality of decisions.",
    "Correct answer for Article 63.",
    "Partially discusses Article 47/state nationalization, but incomplete and mixed.",
    "Partially correct: personal data is processed by law, but KVKK/other laws missing.",
    "Partially captures regulation by law and some judge/prosecutor details, but incomplete.",
    "Does not identify Article 115.",
    "Mentions Information Evaluation Board but lacks application/remedy details.",
    "Invalid sample: question and expected answer are mismatched."
]

df["retrieval_quality"] = retrieval_quality_labels
df["generation_quality"] = generation_quality_labels
df["error_type"] = error_type_labels
df["notes"] = notes

df.head()

,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_context,top1_original_rank,top1_rerank_rank,top1_rerank_score,top1_fusion_score,retrieved_contexts,is_valid_sample,manual_score,retrieval_quality,generation_quality,error_type,notes
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,Sen Türk hukuk metinleri için çalışan dikkatli...,"Anayasanın 101. Maddesiyle ilgili tartışmalar,...",chunk_000210,Türkiye Cumhuriyeti Anayasası,Madde 158 – Uyuşmazlık Mahkemesi adli ve idari...,1,4,0.007053,0.7750,Madde 158 – Uyuşmazlık Mahkemesi adli ve idari...,True,0.0,partial_context,wrong_generation,partial_context,Expected discussion about Article 101 eligibil...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...",Sen Türk hukuk metinleri için çalışan dikkatli...,"Anayasanın 10. Maddesi, herkesin eşittirliği s...",chunk_000188,Türkiye Cumhuriyeti Anayasası,"Madde 150 – Kanunların, Cumhurbaşkanlığı karar...",1,4,0.000305,0.7750,"Madde 150 – Kanunların, Cumhurbaşkanlığı karar...",True,0.5,correct_context,partial_generation,incomplete_answer,"Equality principle is partially mentioned, but..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...",Sen Türk hukuk metinleri için çalışan dikkatli...,"Hayatın sınırlanması, Anayasanın 17. Maddesine...",chunk_000373,Türkiye Cumhuriyeti Anayasası,"Madde 17 – Herkes, yaşama, maddi ve manevi var...",1,1,0.385170,1.0000,"Madde 17 – Herkes, yaşama, maddi ve manevi var...",True,0.5,correct_context,partial_generation,correct_context_wrong_generation,"Mentions life/right limitation, but legal conc..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,Sen Türk hukuk metinleri için çalışan dikkatli...,Geçici madde 20 1987 yılında eklendi.,chunk_000600,Bilgi Edinme Kanunu,Madde 20- Açıklanması veya zamanından önce açı...,1,8,0.000864,0.7375,Madde 20- Açıklanması veya zamanından önce açı...,True,0.0,wrong_context,wrong_generation,wrong_context,"Expected date is 20 May 2016, generated says 1..."
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,Sen Türk hukuk metinleri için çalışan dikkatli...,"Hayır, TCK 121 ihlali sabit değildir. (Madde 1...",chunk_000101,Türkiye Cumhuriyeti Anayasası,Madde 121 – (Mülga: 21/1/2017-6771/16 md.) B. ...,1,6,0.000157,0.7500,Madde 121 – (Mülga: 21/1/2017-6771/16 md.) B. ...,True,0.5,wrong_context,partial_generation,wrong_context_partial_answer,"Generated says violation is not fixed, but ret..."


In [13]:
valid_df = df[df["is_valid_sample"] == True].copy()

print("Error type counts:")
display(valid_df["error_type"].value_counts())

print("\nRetrieval quality counts:")
display(valid_df["retrieval_quality"].value_counts())

print("\nGeneration quality counts:")
display(valid_df["generation_quality"].value_counts())

Error type counts:


,count
error_type,
wrong_context,6
partial_answer,5
correct_context_wrong_generation,3
correct,2
partial_context,1
incomplete_answer,1
wrong_context_partial_answer,1



Retrieval quality counts:


,count
retrieval_quality,
correct_context,7
wrong_context,7
partial_context,5



Generation quality counts:


,count
generation_quality,
wrong_generation,9
partial_generation,8
correct_generation,2


In [14]:
valid_df.groupby("error_type")["manual_score"].agg(["count", "mean"])

,count,mean
error_type,,
correct,2,1.000000
correct_context_wrong_generation,3,0.166667
incomplete_answer,1,0.500000
partial_answer,5,0.500000
partial_context,1,0.000000
wrong_context,6,0.000000
wrong_context_partial_answer,1,0.500000


In [15]:
valid_df.groupby("retrieval_quality")["manual_score"].agg(["count", "mean"])

,count,mean
retrieval_quality,,
correct_context,7,0.500000
partial_context,5,0.400000
wrong_context,7,0.071429


In [16]:
error_summary_df = valid_df["error_type"].value_counts().reset_index()
error_summary_df.columns = ["error_type", "count"]

error_summary_df["percentage"] = (
    error_summary_df["count"] / len(valid_df) * 100
).round(2)

error_summary_df

,error_type,count,percentage
0,wrong_context,6,31.58
1,partial_answer,5,26.32
2,correct_context_wrong_generation,3,15.79
3,correct,2,10.53
4,partial_context,1,5.26
5,incomplete_answer,1,5.26
6,wrong_context_partial_answer,1,5.26


In [17]:
retrieval_summary_df = valid_df["retrieval_quality"].value_counts().reset_index()
retrieval_summary_df.columns = ["retrieval_quality", "count"]

retrieval_summary_df["percentage"] = (
    retrieval_summary_df["count"] / len(valid_df) * 100
).round(2)

retrieval_summary_df

,retrieval_quality,count,percentage
0,correct_context,7,36.84
1,wrong_context,7,36.84
2,partial_context,5,26.32


In [18]:
generation_summary_df = valid_df["generation_quality"].value_counts().reset_index()
generation_summary_df.columns = ["generation_quality", "count"]

generation_summary_df["percentage"] = (
    generation_summary_df["count"] / len(valid_df) * 100
).round(2)

generation_summary_df

,generation_quality,count,percentage
0,wrong_generation,9,47.37
1,partial_generation,8,42.11
2,correct_generation,2,10.53


In [19]:
df.to_csv(
    f"{metrics_path}/turkish_bge_reranker_error_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

error_summary_df.to_csv(
    f"{metrics_path}/turkish_bge_reranker_error_type_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

retrieval_summary_df.to_csv(
    f"{metrics_path}/turkish_bge_reranker_retrieval_quality_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

generation_summary_df.to_csv(
    f"{metrics_path}/turkish_bge_reranker_generation_quality_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Error analysis files saved.")

Error analysis files saved.
